In [7]:
import numpy as np
from scipy.stats import binom
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler

from dataset_generation import (
    poly_add, poly_mul, matvec, random_poly, sample_cbd,
    generate_mlwe_sample, generate_uniform_sample
)
from evaluation import (
    coeff_variance, center_mod_q, variance_threshold,
    variance_distinguisher, cohens_d, binomial_ci, logreg_accuracy
)

seed = 0
rng = np.random.default_rng(seed)

# Base configuration
config = {
    "n": 256,
    "q": 3329,
    "k": 3,
    "eta": 2,
    "rng": rng,
}


# NTT tables (Appendix A)
zetas = np.array([
    1,1729,2580,3289,2642,630,1897,848,
    1062,1919,193,797,2786,3260,569,1746,
    296,2447,1339,1476,3046,56,2240,1333,
    1426,2094,535,2882,2393,2879,1974,821,
    289,331,3253,1756,1197,2304,2277,2055,
    650,1977,2513,632,2865,33,1320,1915,
    2319,1435,807,452,1438,2868,1534,2402,
    2647,2617,1481,648,2474,3110,1227,910,
    17,2761,583,2649,1637,723,2288,1100,
    1409,2662,3281,233,756,2156,3015,3050,
    1703,1651,2789,1789,1847,952,1461,2687,
    939,2308,2437,2388,733,2337,268,641,
    1584,2298,2037,3220,375,2549,2090,1645,
    1063,319,2773,757,2099,561,2466,2594,
    2804,1092,403,1026,1143,2150,2775,886,
    1722,1212,1874,1029,2110,2935,885,2154
], dtype=np.int64)

gammas = np.array([
    17,-17,2761,-2761,583,-583,2649,-2649,
    1637,-1637,723,-723,2288,-2288,1100,-1100,
    1409,-1409,2662,-2662,3281,-3281,233,-233,
    756,-756,2156,-2156,3015,-3015,3050,-3050,
    1703,-1703,1651,-1651,2789,-2789,1789,-1789,
    1847,-1847,952,-952,1461,-1461,2687,-2687,
    939,-939,2308,-2308,2437,-2437,2388,-2388,
    733,-733,2337,-2337,268,-268,641,-641,
    1584,-1584,2298,-2298,2037,-2037,3220,-3220,
    375,-375,2549,-2549,2090,-2090,1645,-1645,
    1063,-1063,319,-319,2773,-2773,757,-757,
    2099,-2099,561,-561,2466,-2466,2594,-2594,
    2804,-2804,1092,-1092,403,-403,1026,-1026,
    1143,-1143,2150,-2150,2775,-2775,886,-886,
    1722,-1722,1212,-1212,1874,-1874,1029,-1029,
    2110,-2110,2935,-2935,885,-885,2154,-2154
], dtype=np.int64) % config["q"]

config["zetas"] = zetas
config["gammas"] = gammas


In [8]:
def sparse_random(config, nz):
    # Baseline: nz nonzero entries placed uniformly at random
    k, n = config["k"], config["n"]
    positions = config["rng"].choice(k*k, nz, replace=False)
    positions = set(positions)
    return [[random_poly(config) if i*k+j in positions
             else np.zeros(n, dtype=int)
             for j in range(k)] for i in range(k)]

def sparse_row_covered(config, nz):
    # Every row has at least one nonzero; remaining nz-k entries placed randomly
    k, n = config["k"], config["n"]
    assert nz >= k, "nz must be >= k to guarantee row coverage"
    
    # Guarantee one nonzero per row
    positions = set()
    for i in range(k):
        j = config["rng"].integers(0, k)
        positions.add(i*k + j)
    
    # Place remaining entries randomly among uncovered positions
    remaining = list(set(range(k*k)) - positions)
    extra = config["rng"].choice(remaining, nz - k, replace=False)
    positions.update(extra)
    
    return [[random_poly(config) if i*k+j in positions
             else np.zeros(n, dtype=int)
             for j in range(k)] for i in range(k)]

def sparse_row_col_covered(config, nz):
    # Every row and column has at least one nonzero
    k, n = config["k"], config["n"]
    assert nz >= k, "nz must be >= k for full row+col coverage"
    
    # Use a random permutation matrix as the coverage seed
    perm = config["rng"].permutation(k)
    positions = set(i*k + perm[i] for i in range(k))
    
    # Place remaining entries randomly
    remaining = list(set(range(k*k)) - positions)
    if nz - k > 0:
        extra = config["rng"].choice(remaining, nz - k, replace=False)
        positions.update(extra)
    
    return [[random_poly(config) if i*k+j in positions
             else np.zeros(n, dtype=int)
             for j in range(k)] for i in range(k)]

def sparse_balanced(config, nz):
    # Row weights differ by at most 1: floor(nz/k) or ceil(nz/k) per row
    k, n = config["k"], config["n"]
    base, extra = divmod(nz, k)
    
    positions = set()
    for i in range(k):
        row_nz = base + (1 if i < extra else 0)
        cols = config["rng"].choice(k, row_nz, replace=False)
        for j in cols:
            positions.add(i*k + j)
    
    return [[random_poly(config) if i*k+j in positions
             else np.zeros(n, dtype=int)
             for j in range(k)] for i in range(k)]

def sparse_diagonal(config, nz):
    # Nonzero entries on diagonal first, then random off-diagonal extras
    k, n = config["k"], config["n"]
    assert nz >= k, "nz must be >= k to fill diagonal"
    
    positions = set(i*k + i for i in range(k))
    remaining = list(set(range(k*k)) - positions)
    if nz - k > 0:
        extra = config["rng"].choice(remaining, nz - k, replace=False)
        positions.update(extra)
    
    return [[random_poly(config) if i*k+j in positions
             else np.zeros(n, dtype=int)
             for j in range(k)] for i in range(k)]

In [9]:
def generate_dataset_structured_sparse(config, A_fn, nz, N=5000):
    A = A_fn(config, nz)
    dataset = []
    for _ in range(N):
        if config["rng"].random() < 0.5:
            _, _, b = generate_mlwe_sample(A, config)
            dataset.append((b, 1))
        else:
            b = generate_uniform_sample(config)
            dataset.append((b, 0))
    return A, dataset

In [10]:
families = [
    ("random",       sparse_random),
    ("row-covered",  sparse_row_covered),
    ("row+col-cov",  sparse_row_col_covered),
    ("balanced",     sparse_balanced),
    ("diagonal",     sparse_diagonal),
]

nz_values = [3, 4, 5, 6]
N = 5000
n_matrices = 10  # average over multiple matrix draws for stability

print(f"{'family':<14} {'nz':>4} {'accuracy':>10} {'ci_lo':>8} {'ci_hi':>8} {'cohen_d':>10}")

structured_results = []

for fname, A_fn in families:
    for nz in nz_values:
        accs, ds = [], []
        
        for _ in range(n_matrices):
            A, dataset = generate_dataset_structured_sparse(config, A_fn, nz, N=N)
            tau = variance_threshold(config)
            
            correct = sum(1 for b, label in dataset
                         if variance_distinguisher(b, tau, config) == label)
            accs.append(correct / N)
            
            d, _, _, _, _ = cohens_d(A, config, n_samples=200)
            ds.append(d)
        
        acc = np.mean(accs)
        lo, hi = binomial_ci(int(acc * N), N)
        d_mean = np.mean(ds)
        
        structured_results.append((fname, nz, acc, lo, hi, d_mean))
        print(f"{fname:<14} {nz:>4} {acc:>10.4f} {lo:>8.4f} {hi:>8.4f} {d_mean:>10.4f}")

family           nz   accuracy    ci_lo    ci_hi    cohen_d


KeyboardInterrupt: 